<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/02)_intent_Classifier_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결 및 확인

from google.colab import drive

drive.mount("/content/drive")

%cd /content/drive/MyDrive/rag_intent_chatbot

!pwd
!find . -maxdepth 3 -type f | sort

Mounted at /content/drive
/content/drive/MyDrive/rag_intent_chatbot
/content/drive/MyDrive/rag_intent_chatbot
./data/documents/sample.txt
./data/intents.json
./interface_check.txt
./main.py
./models/intent_classifier.pt
./__pycache__/main.cpython-312.pyc
./requirements.txt
./src/chatbot.py
./src/__init__.py
./src/intent/dataset.py
./src/intent/__init__.py
./src/intent/model.py
./src/intent/predict.py
./src/intent/train.py
./src/__pycache__/chatbot.cpython-312.pyc
./src/__pycache__/__init__.cpython-312.pyc
./src/rag/chunker.py
./src/rag/document_loader.py
./src/rag/embedder.py
./src/rag/__init__.py
./src/rag/retriever.py
./src/rag/text_preprocessor.py
./src/rag/vector_store.py


In [ ]:
# intent.json이라는 파일을 만든다
# json 파일은 텍스트 파일을 딕셔너리로 바꿔준다
%%writefile data/intents.json
{
  "intents": [
    {
      "tag": "greeting",
      "patterns": [
        "안녕",
        "안녕하세요",
        "반가워",
        "반갑습니다",
        "좋은 아침이야",
        "좋은 오후야",
        "챗봇아 안녕",
        "처음 뵙겠습니다",
        "하이",
        "hello"
      ],
      "responses": [
        "안녕하세요. 무엇을 도와드릴까요?",
        "반갑습니다. 궁금한 내용을 입력해주세요.",
        "안녕하세요. 문서에 관한 질문도 할 수 있습니다."
      ]
    },
    {
      "tag": "goodbye",
      "patterns": [
        "잘 가",
        "안녕히 가세요",
        "다음에 보자",
        "대화를 종료할게",
        "종료",
        "그만할게",
        "이제 갈게",
        "수고했어",
        "bye",
        "대화 끝"
      ],
      "responses": [
        "대화를 종료하겠습니다.",
        "이용해주셔서 감사합니다.",
        "다음에 다시 만나요."
      ]
    },
    {
      "tag": "thanks",
      "patterns": [
        "고마워",
        "감사합니다",
        "도와줘서 고마워",
        "답변 고마워",
        "정말 감사합니다",
        "잘 알려줘서 고마워",
        "도움이 됐어",
        "친절한 설명 고마워",
        "thanks",
        "좋은 답변이야"
      ],
      "responses": [
        "도움이 되었다니 다행입니다.",
        "천만에요.",
        "더 궁금한 내용이 있으면 질문해주세요."
      ]
    },
    {
      "tag": "help",
      "patterns": [
        "사용법을 알려줘",
        "어떻게 사용해",
        "무엇을 할 수 있어",
        "어떤 기능이 있어",
        "질문은 어떻게 해야 해",
        "챗봇 사용 방법",
        "도움말 보여줘",
        "기능을 설명해줘",
        "어떻게 질문하면 돼",
        "지원하는 기능이 뭐야"
      ],
      "responses": [
        "일반적인 대화를 하거나 저장된 문서에 관해 질문할 수 있습니다.",
        "문서 내용을 묻고 싶다면 '문서에서 해당 내용을 찾아줘'와 같이 질문해주세요.",
        "현재는 인텐트 분류 기능을 제공하며, 다음 단계에서 RAG 문서 검색 기능이 연결됩니다."
      ]
    },
    {
      "tag": "bot_info",
      "patterns": [
        "너는 누구야",
        "너는 어떤 챗봇이야",
        "이 챗봇은 뭐야",
        "너의 역할이 뭐야",
        "어떤 모델이야",
        "이 프로젝트를 설명해줘",
        "RAG 챗봇이 뭐야",
        "인텐트 분류기가 뭐야",
        "너는 무엇을 위해 만들어졌어",
        "챗봇 소개해줘"
      ],
      "responses": [
        "저는 PyTorch 인텐트 분류기와 RAG 검색 기능을 결합한 학습용 챗봇입니다.",
        "사용자의 의도를 분류하고, 문서 질문은 RAG 검색기로 전달하는 챗봇입니다.",
        "현재 프로젝트에서는 PyTorch 기반 인텐트 분류와 문서 검색 기능을 구현하고 있습니다."
      ]
    },
    {
      "tag": "document_query",
      "patterns": [
        "문서에서 찾아줘",
        "자료에서 해당 내용을 검색해줘",
        "파일에 어떤 내용이 있어",
        "문서 내용을 알려줘",
        "이 문서를 요약해줘",
        "업로드한 자료에서 답을 찾아줘",
        "문서에 따르면 무엇이야",
        "파일을 검색해줘",
        "저장된 문서에서 찾아봐",
        "자료를 기반으로 답해줘",
        "sample.txt 내용을 알려줘",
        "문서에 나온 내용을 설명해줘",
        "관련 자료가 있는지 찾아줘",
        "문서에서 핵심 내용을 가져와줘"
      ],
      "responses": [
        "문서 검색이 필요한 질문으로 판단했습니다.",
        "저장된 문서에서 관련 내용을 검색하겠습니다.",
        "이 질문은 RAG 검색기로 전달할 수 있습니다."
      ]
    }
  ]
}

Overwriting data/intents.json


In [ ]:
# 저장된 내용 확인
!python -m json.tool data/intents.json

{
    "intents": [
        {
            "tag": "greeting",
            "patterns": [
                "\uc548\ub155",
                "\uc548\ub155\ud558\uc138\uc694",
                "\ubc18\uac00\uc6cc",
                "\ubc18\uac11\uc2b5\ub2c8\ub2e4",
                "\uc88b\uc740 \uc544\uce68\uc774\uc57c",
                "\uc88b\uc740 \uc624\ud6c4\uc57c",
                "\ucc57\ubd07\uc544 \uc548\ub155",
                "\ucc98\uc74c \ubd59\uaca0\uc2b5\ub2c8\ub2e4",
                "\ud558\uc774",
                "hello"
            ],
            "responses": [
                "\uc548\ub155\ud558\uc138\uc694. \ubb34\uc5c7\uc744 \ub3c4\uc640\ub4dc\ub9b4\uae4c\uc694?",
                "\ubc18\uac11\uc2b5\ub2c8\ub2e4. \uad81\uae08\ud55c \ub0b4\uc6a9\uc744 \uc785\ub825\ud574\uc8fc\uc138\uc694.",
                "\uc548\ub155\ud558\uc138\uc694. \ubb38\uc11c\uc5d0 \uad00\ud55c \uc9c8\ubb38\ub3c4 \ud560 \uc218 \uc788\uc2b5\ub2c8\ub2e4."
            ]
        },
        {
            "

In [ ]:
%%writefile src/intent/dataset.py
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset


# 한글, 영어, 숫자만 토큰으로 추출합니다
TOKEN_PATTERN = re.compile(r"[가-힣]+|[a-zA-Z]+|[0-9]+")


def tokenize(text: str) -> List[str]:
    """
    문장을 한글, 영어, 숫자 단위의 토큰으로 분리합니다.

    예시:
        "문서에서 RAG 내용을 찾아줘!"
        -> ["문서에서", "rag", "내용을", "찾아줘"]
    """
    if not isinstance(text, str): # text가 str이면 True, 아니면 False 반환
        raise TypeError("text는 문자열이어야 합니다.")

    return TOKEN_PATTERN.findall(text.lower()) # findall(무엇을, 어디서)


def load_intents(json_path: str | Path) -> Dict:
    """
    intents.json 파일을 읽고 기본 구조를 검사합니다.
    """
    json_path = Path(json_path) # 문자열 객체를 Path(경로) 객체로 변환

    if not json_path.exists(): # 파일 존재하지 않는 경우
        raise FileNotFoundError(f"인텐트 파일을 찾을 수 없습니다: {json_path}")

    with json_path.open("r", encoding="utf-8") as file: # 파일을 열어서 file로 지정
        data = json.load(file) # 리스트 안에 딕셔너리 형태로 파일을 바꿔준다

        # {intents : [{"tag": "greeting", "patterns":       }] 형식임 (하나의 키를 가진 딕셔너리)

    if "intents" not in data:
        raise ValueError("JSON 최상위에 'intents' 키가 필요합니다.")

    if not isinstance(data["intents"], list):
        raise ValueError("'intents'는 리스트 형식이어야 합니다.")

    required_keys = {"tag", "patterns", "responses"} # set

    for index, intent in enumerate(data["intents"]): # enumerate는 ((인덱스, 내용물)()...) 로 바꿔준다.
        missing_keys = required_keys - set(intent.keys())

        if missing_keys: # 만약 필수 키가 intent에 존재하지 않으면
            raise ValueError(
                f"{index}번째 인텐트에 필수 키가 없습니다: {missing_keys}"
            )

        if not intent["patterns"]: # intent["patterns"] 가 비어있으면
            raise ValueError(
                f"'{intent['tag']}' 인텐트의 patterns가 비어 있습니다."
            )

    return data # json을 후처리 한 파일


def build_vocabulary(
    texts: Sequence[str], # Sequence[]는 순서가 존재하면 상관 없다는 뜻(set, dict는 순서 없기에 안됨)
    min_frequency: int = 1
) -> List[str]:
    """
    여러 문장에서 전체 단어 사전을 생성합니다.

    min_frequency:
        최소 등장 횟수입니다.
        1이면 한 번 이상 등장한 모든 토큰을 사용합니다.
    """
    if min_frequency < 1:
        raise ValueError("min_frequency는 1 이상이어야 합니다.")

    token_counter: Counter[str] = Counter() # 빈 딕셔너리를 만든다
    # Counter을 사용하는 이유
    # 딕셔너리 내에 존재하지 않는 키를 넣어줘도 0 반환한다
    # 내장 기능이 많다(update, most_common, 사칙연산 가능)
    # 의도가 명확하다

    for text in texts:
        token_counter.update(tokenize(text))

    vocabulary = sorted(
        token
        for token, count in token_counter.items()
        if count >= min_frequency
    )
    # 지정한 빈도수(기본적으로 1) 이상인 key들만 정렬한다(가나다, 알파벳 순)

    if not vocabulary:
        raise ValueError("생성된 단어 사전이 비어 있습니다.")

    return vocabulary


def bag_of_words(
    text: str,
    vocabulary_to_index: Dict[str, int] #
) -> np.ndarray:
    """
    문장을 Bag-of-Words 벡터로 변환합니다.

    단어가 문장에 존재하면 1, 존재하지 않으면 0으로 표시합니다.
    """
    vector = np.zeros(len(vocabulary_to_index), dtype=np.float32) # len(vocabulary_to_index) 만큼의 0.0을 생성
    tokens = set(tokenize(text)) # 겹치는 것 제거

    for token in tokens:
        token_index = vocabulary_to_index.get(token) # dict 내 value 꺼내기

        if token_index is not None: # get()은 존재하지 않을 경우 None을 반환한다. 즉, 존재하는 경우에 해당
            vector[token_index] = 1.0 # 해당하는 index를 1.0으로 바꾼다

    return vector # 존재하는 것을 1.0으로, 존재하지 않는 것을 0.0으로 표기


class IntentDataset(Dataset):
    """
    PyTorch Intent Classifier 학습용 Dataset입니다.
    """

    def __init__(
        self,
        samples: Sequence[Tuple[str, int]],
        vocabulary: Sequence[str]
    ) -> None:
        if not samples:
            raise ValueError("학습 샘플이 비어 있습니다.")

        if not vocabulary:
            raise ValueError("단어 사전이 비어 있습니다.")

        self.samples = list(samples)
        self.vocabulary = list(vocabulary)
        self.vocabulary_to_index = {
            token: index
            for index, token in enumerate(self.vocabulary)
        }

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        text, label = self.samples[index]

        input_vector = bag_of_words(
            text=text,
            vocabulary_to_index=self.vocabulary_to_index
        )

        input_tensor = torch.from_numpy(input_vector) # numpy 배열을 pytorch 배열로 변환
        label_tensor = torch.tensor(label, dtype=torch.long) # 정수형 tensor로 새로 만듦(정답 레이블)

        return input_tensor, label_tensor


def prepare_training_data(
    json_path: str | Path,
    min_frequency: int = 1
) -> Tuple[IntentDataset, List[str], List[str], Dict[str, List[str]]]:
    """
    intents.json을 읽어서 학습에 필요한 객체를 생성합니다.

    반환값:
        dataset:
            PyTorch Dataset

        vocabulary:
            Bag-of-Words에 사용하는 단어 목록

        tag_names:
            인텐트 이름 목록

        response_map:
            인텐트별 응답 문장
    """
    data = load_intents(json_path)

    all_patterns: List[str] = []
    raw_samples: List[Tuple[str, str]] = []
    response_map: Dict[str, List[str]] = {}

    for intent in data["intents"]:
        tag = intent["tag"]
        response_map[tag] = intent["responses"] # response_map = {tag : responses}

        for pattern in intent["patterns"]:
            all_patterns.append(pattern) # all_patterns = [patterns]
            raw_samples.append((pattern, tag)) # raw_samples = [(patterns, tag)]



    tag_names = sorted({tag for _, tag in raw_samples}) # raw_samples의 tag만 반환 후 정렬
    tag_to_index = {
        tag: index
        for index, tag in enumerate(tag_names)
    }  # tag_to_index = {tag : index}

    vocabulary = build_vocabulary(
        texts=all_patterns,
        min_frequency=min_frequency
    )

    indexed_samples = [
        (text, tag_to_index[tag])
        for text, tag in raw_samples
    ] # indexed samples = [(patterns, index)]

    dataset = IntentDataset(
        samples=indexed_samples,
        vocabulary=vocabulary
    )

    return dataset, vocabulary, tag_names, response_map

Overwriting src/intent/dataset.py


In [ ]:
# 동작 확인

from src.intent.dataset import (
    tokenize,
    prepare_training_data
)

print(tokenize("문서에서 RAG 내용을 찾아줘!"))

dataset, vocabulary, tags, responses = prepare_training_data(
    "data/intents.json"
)

print("학습 문장 수:", len(dataset))
print("단어 사전 크기:", len(vocabulary))
print("인텐트:", tags)

['문서에서', 'rag', '내용을', '찾아줘']
학습 문장 수: 64
단어 사전 크기: 110
인텐트: ['bot_info', 'document_query', 'goodbye', 'greeting', 'help', 'thanks']


In [ ]:
# intent 분류
# 구조는 입력층 -> linear -> Relu -> Dropout -> Linear -> Relu -> Dropout -> 출력층 이다.
# 뉴런의 수는 인텐트의 개수와 같다

In [ ]:
%%writefile src/intent/model.py
from __future__ import annotations

import torch
from torch import nn


class IntentClassifier(nn.Module):
    """
    Bag-of-Words 벡터를 입력받아 인텐트를 분류하는 다층 신경망입니다.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        dropout: float = 0.2
    ) -> None:
        super().__init__() # nn.Module 부모 클래스를 상속받음

        if input_size <= 0:
            raise ValueError("input_size는 1 이상이어야 합니다.")

        if hidden_size <= 1:
            raise ValueError("hidden_size는 2 이상이어야 합니다.")

        if output_size <= 1:
            raise ValueError("output_size는 2 이상이어야 합니다.")

        if not 0.0 <= dropout < 1.0:
            raise ValueError("dropout은 0 이상 1 미만이어야 합니다.")

        second_hidden_size = max(hidden_size // 2, 2)

        self.network = nn.Sequential( # 예측값 반환
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size, second_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(second_hidden_size, output_size)
        ) # 이때 입력값은 반드시 전처리한 Tensor 형태여야한다.

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        '''
        이때 forward는 특별히 호출하지 않아도 작동한다
        예를 들어, model = IntentClassifier(        )
        model(x) -> forward(x)
        의 형태로 작동한다(nn.Module내에 내장된 기능)
        '''
        """
        Softmax는 여기에서 사용하지 않습니다.

        Pytorch의 CrossEntropyLoss 내부에서 필요한 확률 계산을 처리하므로
        모델은 원본 출력값인 logits를 반환합니다.(Softmax 사용시 2번 사용하는 것)
        """
        return self.network(inputs)

Overwriting src/intent/model.py


In [ ]:
import torch

from src.intent.model import IntentClassifier

test_model = IntentClassifier(
    input_size=100,
    hidden_size=64,
    output_size=6
)

dummy_input = torch.randn(4, 100)
dummy_output = test_model(dummy_input)

print(dummy_output.shape)

torch.Size([4, 6])


In [ ]:
%%writefile src/intent/train.py
from __future__ import annotations

import argparse
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from .dataset import IntentDataset, prepare_training_data
from .model import IntentClassifier


def set_seed(seed: int) -> None:
    """
    학습 결과를 최대한 재현할 수 있도록 난수 시드를 고정합니다.
    """
    random.seed(seed) # 데이터 분할 고정(코드를 다시 실행해도 결과가 바뀌지 않게 하기 위함)
    np.random.seed(seed) # 데이터 전처리 고정
    torch.manual_seed(seed) # 모델 학습/초기화 고정

    # 모든 시행에서의 랜덤한 값들을 고정



    if torch.cuda.is_available(): # GPU 사용 가능 여부
        torch.cuda.manual_seed_all(seed)


def stratified_split_indices(
    dataset: IntentDataset, # class IntentDataset 를 받을 예정
    validation_ratio: float,
    seed: int
) -> Tuple[List[int], List[int]]:
    """
    각 인텐트의 문장이 학습과 검증 데이터에 모두 포함되도록
    클래스별로 데이터를 나눕니다.
    """
    if not 0.0 <= validation_ratio < 1.0:
        raise ValueError("validation_ratio는 0 이상 1 미만이어야 합니다.")

    label_to_indices: Dict[int, List[int]] = defaultdict(list) # key를 선언하면 value를 list 형태로 받는다

    for sample_index, (_, label) in enumerate(dataset.samples): # .samples는 IntentDataset 내에 있는 함수
        label_to_indices[label].append(sample_index)
        # 각각의 라벨이 어느 인덱스에서 나타났는지 딕셔너리 형태로 저장

    random_generator = random.Random(seed)

    train_indices: List[int] = []
    validation_indices: List[int] = []

    for indices in label_to_indices.values(): # list(어느 인덱스에서 나타났는지)
        random_generator.shuffle(indices)

        if validation_ratio == 0.0 or len(indices) < 2:# 검증데이터로 쓸 개수 계산
            validation_count = 0
            # 검증 데이터 0개
        else:
            validation_count = max(
                1, # 최소 1개는 검증용으로서 존재해야함
                round(len(indices) * validation_ratio) # 반올림
            )

            validation_count = min(
                validation_count,
                len(indices) - 1 # 최소 1개는 학습용으로 남겨놓기 위함
            )


        validation_indices.extend(indices[:validation_count])
        train_indices.extend(indices[validation_count:])


    # 다시 섞어서 반환
    random_generator.shuffle(validation_indices)
    random_generator.shuffle(train_indices)

    return train_indices, validation_indices


def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> Tuple[float, float]:
    """
    검증 데이터의 평균 손실과 정확도를 계산합니다.
    """
    model.eval() # 평가 모드로 전환(학습 모드의 경우 과적합을 막기 위해 일부 뉴런을 랜덤하게 끈다.)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad(): # 기울기 계산 모드 끄기(메모리/속도 최적화)
        for inputs, labels in data_loader: # GPU 사용 가능한 경우(아닌 경우에는 전부 CPU에서 진행한다)
            inputs = inputs.to(device) # 입력 데이터 GPU 이동
            labels = labels.to(device) # 정답 라벨 GPU 이동


            #순전파

            logits = model(inputs)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()
            total_samples += batch_size

    if total_samples == 0:
        return 0.0, 0.0

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy


def save_checkpoint(
    output_path: Path,
    model: IntentClassifier,
    vocabulary: Sequence[str],
    tags: Sequence[str],
    responses: Dict[str, List[str]],
    hidden_size: int,
    dropout: float,
    epoch: int,
    validation_loss: float,
    validation_accuracy: float
) -> None:
    """
    추론에 필요한 모델과 메타데이터를 하나의 파일로 저장합니다.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "input_size": len(vocabulary),
        "hidden_size": hidden_size,
        "output_size": len(tags),
        "dropout": dropout,
        "vocabulary": list(vocabulary),
        "tags": list(tags),
        "responses": responses,
        "epoch": epoch,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "preprocessing": "regex_bag_of_words_v1"
    }

    torch.save(checkpoint, output_path)


def train(args: argparse.Namespace) -> None:
    set_seed(args.seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"사용 장치: {device}")

    dataset, vocabulary, tags, responses = prepare_training_data(
        json_path=args.data_path,
        min_frequency=args.min_frequency
    )

    train_indices, validation_indices = stratified_split_indices(
        dataset=dataset,
        validation_ratio=args.validation_ratio,
        seed=args.seed
    )

    train_dataset = Subset(dataset, train_indices)
    validation_dataset = Subset(dataset, validation_indices)

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=device.type == "cuda"
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=device.type == "cuda"
    )

    model = IntentClassifier(
        input_size=len(vocabulary),
        hidden_size=args.hidden_size,
        output_size=len(tags),
        dropout=args.dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay
    )

    output_path = Path(args.output_path)

    best_validation_loss = float("inf")
    best_validation_accuracy = 0.0
    best_epoch = 0
    epochs_without_improvement = 0

    print(f"전체 샘플 수: {len(dataset)}")
    print(f"학습 샘플 수: {len(train_dataset)}")
    print(f"검증 샘플 수: {len(validation_dataset)}")
    print(f"단어 사전 크기: {len(vocabulary)}")
    print(f"인텐트 수: {len(tags)}")
    print(f"인텐트 목록: {tags}")
    print("-" * 60)

    for epoch in range(1, args.epochs + 1):
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_sample_count = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(inputs)
            loss = criterion(logits, labels)

            loss.backward()

            # 지나치게 큰 gradient를 방지합니다.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            optimizer.step()

            batch_size = labels.size(0)

            train_loss_sum += loss.item() * batch_size
            train_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()
            train_sample_count += batch_size

        train_loss = train_loss_sum / train_sample_count
        train_accuracy = train_correct / train_sample_count

        validation_loss, validation_accuracy = evaluate(
            model=model,
            data_loader=validation_loader,
            criterion=criterion,
            device=device
        )

        is_improved = validation_loss < best_validation_loss

        if is_improved:
            best_validation_loss = validation_loss
            best_validation_accuracy = validation_accuracy
            best_epoch = epoch
            epochs_without_improvement = 0

            save_checkpoint(
                output_path=output_path,
                model=model,
                vocabulary=vocabulary,
                tags=tags,
                responses=responses,
                hidden_size=args.hidden_size,
                dropout=args.dropout,
                epoch=epoch,
                validation_loss=validation_loss,
                validation_accuracy=validation_accuracy
            )
        else:
            epochs_without_improvement += 1

        should_print = (
            epoch == 1
            or epoch % args.print_every == 0
            or epoch == args.epochs
        )

        if should_print:
            print(
                f"Epoch {epoch:03d}/{args.epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_accuracy:.2%} | "
                f"Val Loss: {validation_loss:.4f} | "
                f"Val Acc: {validation_accuracy:.2%}"
            )

        if epochs_without_improvement >= args.patience:
            print(
                f"\n검증 손실이 {args.patience}회 동안 개선되지 않아 "
                "학습을 조기 종료합니다."
            )
            break

    print("-" * 60)
    print(f"최적 Epoch: {best_epoch}")
    print(f"최적 검증 손실: {best_validation_loss:.4f}")
    print(f"최적 검증 정확도: {best_validation_accuracy:.2%}")
    print(f"모델 저장 위치: {output_path.resolve()}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="PyTorch Intent Classifier 학습"
    )

    parser.add_argument(
        "--data-path",
        type=str,
        default="data/intents.json"
    )

    parser.add_argument(
        "--output-path",
        type=str,
        default="models/intent_classifier.pt"
    )

    parser.add_argument(
        "--epochs",
        type=int,
        default=300
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=8
    )

    parser.add_argument(
        "--hidden-size",
        type=int,
        default=64
    )

    parser.add_argument(
        "--dropout",
        type=float,
        default=0.2
    )

    parser.add_argument(
        "--learning-rate",
        type=float,
        default=0.001
    )

    parser.add_argument(
        "--weight-decay",
        type=float,
        default=0.0001
    )

    parser.add_argument(
        "--validation-ratio",
        type=float,
        default=0.2
    )

    parser.add_argument(
        "--min-frequency",
        type=int,
        default=1
    )

    parser.add_argument(
        "--patience",
        type=int,
        default=50
    )

    parser.add_argument(
        "--print-every",
        type=int,
        default=20
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42
    )

    return parser.parse_args()


if __name__ == "__main__":
    train(parse_args())

Overwriting src/intent/train.py


In [ ]:
# 학습 실행

%cd /content/drive/MyDrive/rag_intent_chatbot

!python -m src.intent.train

!ls -lh models

/content/drive/MyDrive/rag_intent_chatbot
사용 장치: cpu
전체 샘플 수: 64
학습 샘플 수: 51
검증 샘플 수: 13
단어 사전 크기: 110
인텐트 수: 6
인텐트 목록: ['bot_info', 'document_query', 'goodbye', 'greeting', 'help', 'thanks']
------------------------------------------------------------
Epoch 001/300 | Train Loss: 1.7898 | Train Acc: 13.73% | Val Loss: 1.7909 | Val Acc: 7.69%
Epoch 020/300 | Train Loss: 1.1948 | Train Acc: 66.67% | Val Loss: 1.5848 | Val Acc: 46.15%
Epoch 040/300 | Train Loss: 0.2490 | Train Acc: 100.00% | Val Loss: 1.3052 | Val Acc: 61.54%
Epoch 060/300 | Train Loss: 0.0570 | Train Acc: 100.00% | Val Loss: 1.2951 | Val Acc: 61.54%
Epoch 080/300 | Train Loss: 0.0172 | Train Acc: 100.00% | Val Loss: 1.3484 | Val Acc: 61.54%
Epoch 100/300 | Train Loss: 0.0123 | Train Acc: 100.00% | Val Loss: 1.3227 | Val Acc: 61.54%

검증 손실이 50회 동안 개선되지 않아 학습을 조기 종료합니다.
------------------------------------------------------------
최적 Epoch: 54
최적 검증 손실: 1.2639
최적 검증 정확도: 61.54%
모델 저장 위치: /content/drive/MyDrive/rag_intent_ch

In [ ]:
# intent 예측

%%writefile src/intent/predict.py
from __future__ import annotations

import argparse
import json
import random
from pathlib import Path
from typing import Any, Dict, List

import torch

from .dataset import bag_of_words
from .model import IntentClassifier


def load_torch_checkpoint(
    model_path: str | Path,
    device: torch.device
) -> Dict[str, Any]:
    """
    PyTorch 버전 차이를 고려하여 체크포인트를 불러옵니다.
    """
    try:
        return torch.load(
            model_path,
            map_location=device,
            weights_only=False
        )
    except TypeError:
        return torch.load(
            model_path,
            map_location=device
        )


class IntentPredictor:
    """
    학습된 인텐트 분류기를 불러와 문장의 인텐트를 예측합니다.
    """

    def __init__(
        self,
        model_path: str | Path,
        device: str | None = None
    ) -> None:
        self.model_path = Path(model_path)

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"학습된 모델을 찾을 수 없습니다: {self.model_path}\n"
                "먼저 `python -m src.intent.train`을 실행해주세요."
            )

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        self.device = torch.device(device) # 데이터 처리할 장치에 따라 달라짐

        checkpoint = load_torch_checkpoint(
            model_path=self.model_path,
            device=self.device
        )

        required_keys = {
            "model_state_dict",
            "input_size",
            "hidden_size",
            "output_size",
            "dropout",
            "vocabulary",
            "tags",
            "responses"
        }

        missing_keys = required_keys - set(checkpoint.keys())

        if missing_keys:
            raise ValueError(
                f"체크포인트에 필요한 정보가 없습니다: {missing_keys}"
            )

        self.vocabulary: List[str] = checkpoint["vocabulary"]
        self.tags: List[str] = checkpoint["tags"]
        self.responses: Dict[str, List[str]] = checkpoint["responses"]

        self.vocabulary_to_index = {
            token: index
            for index, token in enumerate(self.vocabulary)
        }

        self.model = IntentClassifier( # 이때의 값은 무작위 가중치로 채운다
            input_size=checkpoint["input_size"],
            hidden_size=checkpoint["hidden_size"],
            output_size=checkpoint["output_size"],
            dropout=checkpoint["dropout"]
        ).to(self.device)

        self.model.load_state_dict(checkpoint["model_state_dict"]) # 학습한 가중치들로 채운다.
        self.model.eval()

    @torch.no_grad() # 미분값을 계산/기록하지 말 것(어차피 검증 모드이니까)
    def predict(
        self,
        text: str,
        threshold: float = 0.60,
        top_k: int = 3
    ) -> Dict[str, Any]:
        """
        문장의 인텐트를 예측합니다.

        threshold:
            최고 확률이 이 값보다 낮으면 fallback으로 처리합니다.

        top_k:
            확률이 높은 인텐트를 몇 개까지 표시할지 설정합니다.
        """
        if not text or not text.strip():
            return {
                "text": text,
                "intent": "fallback",
                "confidence": 0.0,
                "response": "질문을 입력해주세요.",
                "top_predictions": [],
                "requires_rag": False
            }

        if not 0.0 <= threshold <= 1.0:
            raise ValueError("threshold는 0 이상 1 이하여야 합니다.")

        input_vector = bag_of_words(
            text=text,
            vocabulary_to_index=self.vocabulary_to_index
        )

        input_tensor = torch.from_numpy(input_vector)
        input_tensor = input_tensor.unsqueeze(0).to(self.device)

        logits = self.model(input_tensor)
        probabilities = torch.softmax(logits, dim=1).squeeze(0)

        top_k = max(1, min(top_k, len(self.tags)))

        top_probabilities, top_indices = torch.topk(
            probabilities,
            k=top_k
        )

        top_predictions = []

        for probability, tag_index in zip(
            top_probabilities.tolist(),
            top_indices.tolist()
        ):
            top_predictions.append({
                "intent": self.tags[tag_index],
                "confidence": round(probability, 4)
            })

        best_probability = top_probabilities[0].item()
        best_tag = self.tags[top_indices[0].item()]

        if best_probability < threshold:
            predicted_intent = "fallback"
            response = (
                "질문의 의도를 확실하게 판단하지 못했습니다. "
                "조금 더 구체적으로 질문해주세요."
            )
        else:
            predicted_intent = best_tag
            candidate_responses = self.responses.get(best_tag, [])

            if candidate_responses:
                response = random.choice(candidate_responses)
            else:
                response = "해당 요청을 처리하겠습니다."

        return {
            "text": text,
            "intent": predicted_intent,
            "confidence": round(best_probability, 4),
            "response": response,
            "top_predictions": top_predictions,
            "requires_rag": predicted_intent == "document_query"
        }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="학습된 Intent Classifier로 문장 예측"
    )

    parser.add_argument(
        "--model-path",
        type=str,
        default="models/intent_classifier.pt"
    )

    parser.add_argument(
        "--text",
        type=str,
        default=None
    )

    parser.add_argument(
        "--threshold",
        type=float,
        default=0.60
    )

    parser.add_argument(
        "--top-k",
        type=int,
        default=3
    )

    return parser.parse_args()


def main() -> None:
    args = parse_args()

    predictor = IntentPredictor(args.model_path)

    if args.text is not None:
        result = predictor.predict(
            text=args.text,
            threshold=args.threshold,
            top_k=args.top_k
        )

        print(
            json.dumps(
                result,
                ensure_ascii=False,
                indent=2
            )
        )

        return

    print("인텐트 분류기 대화 모드입니다.")
    print("'exit' 또는 'quit'을 입력하면 종료됩니다.")

    while True:
        user_input = input("\n사용자: ").strip()

        if user_input.lower() in {"exit", "quit"}:
            print("프로그램을 종료합니다.")
            break

        result = predictor.predict(
            text=user_input,
            threshold=args.threshold,
            top_k=args.top_k
        )

        print(
            json.dumps(
                result,
                ensure_ascii=False,
                indent=2
            )
        )


if __name__ == "__main__":
    main()

Overwriting src/intent/predict.py


In [ ]:
# 단일 문장 예측

!python -m src.intent.predict --text "안녕하세요 반가워요"

{
  "text": "안녕하세요 반가워요",
  "intent": "greeting",
  "confidence": 0.9207,
  "response": "안녕하세요. 문서에 관한 질문도 할 수 있습니다.",
  "top_predictions": [
    {
      "intent": "greeting",
      "confidence": 0.9207
    },
    {
      "intent": "goodbye",
      "confidence": 0.0228
    },
    {
      "intent": "thanks",
      "confidence": 0.0186
    }
  ],
  "requires_rag": false
}
